In [ ]:
!pip install -q tifffile pillow joblib efficientnet_pytorch==0.7.1


In [ ]:
import glob
import os
import shutil
import subprocess
import sys

REPO = 'https://github.com/Shashaboii/AIMI_Panda_Challenge.git'
BRANCH = 'main'
REPO_DIR = '/kaggle/working/repo'

N_TILES = 36
TILE_SIZE = 192
TILE_FORMAT = 'png'
TRAIN_FOLD = 0
TRAIN_BACKBONE = 'efficientnet-b0'
TRAIN_LOSS = 'ordinal'
TRAIN_EPOCHS = 6
TRAIN_BATCH_SIZE = 2
TRAIN_NUM_WORKERS = 0

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, REPO_DIR], check=True)

PYTHON = sys.executable
subprocess.run(
    [
        PYTHON, '-m', 'pip', 'install', '-q',
        'tifffile', 'pillow', 'joblib', 'efficientnet_pytorch==0.7.1',
    ],
    check=True,
)

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA is not available in this Kaggle session. Turn on a GPU accelerator '
        '(for example T4 x1) in Session options, then restart and rerun the notebook.'
    )
print('CUDA device =', torch.cuda.get_device_name(0))

FOLDS_CSV = os.path.join(REPO_DIR, 'data', 'train_folds.csv')
OUT_DIR = f'/kaggle/working/panda_tiles_{N_TILES}x{TILE_SIZE}_{TILE_FORMAT}'
TRAIN_FEATURE_TAG = f'tiles{N_TILES}_imsize{TILE_SIZE}'
EXPECTED_WEIGHT = f"{TRAIN_BACKBONE.replace('-', '')}_{TRAIN_FEATURE_TAG}_{TRAIN_LOSS}_fold{TRAIN_FOLD}.pth"

def find_slide_dir():
    candidates = [
        '/kaggle/input/prostate-cancer-grade-assessment/train_images',
        '/kaggle/input/competitions/prostate-cancer-grade-assessment/train_images',
    ]
    candidates += sorted(set(glob.glob('/kaggle/input/**/train_images', recursive=True)))
    seen = set()
    for path_value in candidates:
        if path_value in seen or not os.path.isdir(path_value):
            continue
        seen.add(path_value)
        if glob.glob(os.path.join(path_value, '*.tif')) or glob.glob(os.path.join(path_value, '*.tiff')):
            return path_value, 'raw_tiff'
        if glob.glob(os.path.join(path_value, '*.png')):
            return path_value, 'png_fallback'
    return None, None

def count_tile_artifacts(path_value):
    try:
        names = os.listdir(path_value)
    except OSError:
        return 0
    return sum(name.lower().endswith(('.png', '.npy', '.npz')) for name in names)

def find_existing_tile_dir(min_count=1000):
    candidates = []
    for root, _, files in os.walk('/kaggle/input'):
        count = sum(name.lower().endswith(('.png', '.npy', '.npz')) for name in files)
        if count >= min_count:
            candidates.append((root, count))
    if os.path.isdir(OUT_DIR):
        out_count = count_tile_artifacts(OUT_DIR)
        if out_count >= min_count:
            candidates.append((OUT_DIR, out_count))
    if not candidates:
        return None, 0
    candidates.sort(key=lambda item: (0 if 'tile' in item[0].lower() else 1, -item[1], item[0]))
    return candidates[0]

SLIDES_DIR, SLIDE_SOURCE = find_slide_dir()
EXISTING_TILE_DIR, EXISTING_TILE_COUNT = find_existing_tile_dir()
TRAIN_TILE_DIR = EXISTING_TILE_DIR or OUT_DIR

print('REPO_DIR            =', REPO_DIR)
print('SLIDES_DIR          =', SLIDES_DIR)
print('SLIDE_SOURCE        =', SLIDE_SOURCE)
print('EXISTING_TILE_DIR   =', EXISTING_TILE_DIR)
print('EXISTING_TILE_COUNT =', EXISTING_TILE_COUNT)
print('TRAIN_TILE_DIR      =', TRAIN_TILE_DIR)
print('OUT_DIR             =', OUT_DIR)
print('FOLDS_CSV           =', FOLDS_CSV)
print('EXPECTED_WEIGHT     =', EXPECTED_WEIGHT)
print('WORKING_FREE_GIB    =', f"{shutil.disk_usage('/kaggle/working').free / (1024 ** 3):.1f}")

if EXISTING_TILE_DIR is not None:
    print('INFO: using existing tile dataset; tile-build cell will no-op.')
elif SLIDES_DIR is not None:
    print('INFO: no existing tile dataset found; next cell will build tiles into /kaggle/working.')
else:
    input_roots = sorted(glob.glob('/kaggle/input/*'))
    raise RuntimeError(
        'Could not find either a tile dataset or a slide dataset. '
        'Attach a published tile dataset, or attach PANDA slides so this notebook can build tiles. '
        f'Visible /kaggle/input entries: {input_roots[:20]}'
    )


In [ ]:
import subprocess

if EXISTING_TILE_DIR is not None:
    print('Using existing tile dataset at', EXISTING_TILE_DIR)
elif SLIDES_DIR is None:
    raise RuntimeError('No slide dataset attached, and no existing tile dataset was found.')
else:
    cmd = [
        PYTHON, 'scripts/preprocess_tiles.py',
        '--slides-dir', SLIDES_DIR,
        '--output-dir', OUT_DIR,
        '--folds-csv', FOLDS_CSV,
        '--tile-size', str(TILE_SIZE),
        '--n-tiles', str(N_TILES),
        '--level', '1',
        '--format', TILE_FORMAT,
        '--n-jobs', '2',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_DIR, check=True)

TRAIN_TILE_DIR = EXISTING_TILE_DIR or OUT_DIR
print('TRAIN_TILE_DIR =', TRAIN_TILE_DIR)
print('Tile artifact count =', count_tile_artifacts(TRAIN_TILE_DIR))


In [ ]:
import os
import subprocess

if not os.path.isdir(TRAIN_TILE_DIR):
    raise RuntimeError(f'Tile directory does not exist: {TRAIN_TILE_DIR}')

artifact_count = count_tile_artifacts(TRAIN_TILE_DIR)
if artifact_count == 0:
    raise RuntimeError(f'No tile artifacts found in {TRAIN_TILE_DIR}')

cmd = [
    PYTHON, '-m', 'src.train',
    '--fold', str(TRAIN_FOLD),
    '--folds-csv', FOLDS_CSV,
    '--tile-dir', TRAIN_TILE_DIR,
    '--backbone', TRAIN_BACKBONE,
    '--loss', TRAIN_LOSS,
    '--n-tiles', str(N_TILES),
    '--tile-size', str(TILE_SIZE),
    '--epochs', str(TRAIN_EPOCHS),
    '--batch-size', str(TRAIN_BATCH_SIZE),
    '--num-workers', str(TRAIN_NUM_WORKERS),
    '--feature-tag', TRAIN_FEATURE_TAG,
    '--amp',
    '--no-pin-memory',
    '--output-dir', '/kaggle/working',
]
print('Training tile dir:', TRAIN_TILE_DIR)
print('Tile artifact count:', artifact_count)
print('Expected weight:', EXPECTED_WEIGHT)
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)
